In [ ]:
"""
Collapse long-format (bridge-year) NBI panel data to ONE ROW PER BRIDGE,
producing a standard right-censored (event, time) target for sksurv's
RandomSurvivalForest.

sksurv's RandomSurvivalForest does NOT support counting-process / start-stop
interval data -- it wants exactly one (event, time) pair per subject. This
script implements that "Option A" structure:

- time  = bridge_age at the survey where the bridge FIRST hit "poor"
          (event == 1), OR bridge_age at the LAST observed survey if the
          bridge never transitioned (right-censored)
- event = 1 if the bridge ever transitioned to poor, 0 otherwise
- All other covariates (including time-varying ones like climate and
  maintenance) are taken from THAT SAME ROW, i.e. the row at event,
  or the row at last observation if censored. This is the most defensible
  default: it reflects the bridge's state right before/at the outcome,
  rather than averaging conditions across decades of very different risk.

bridge_age is built purely from YEAR_BUILT_027 per your confirmation.
Reconstruction is captured separately via `ever_reconstructed` and
`years_since_reconstruction`, which are kept as ordinary covariates
(not used to redefine the time clock).
"""

import pandas as pd
import numpy as np

# ---- 1. Load your panel (bridge-year) data ----
# df = pd.read_csv("your_bridge_panel.csv")
df = pd.read_csv("massachusetts_bridges_rsf_ready.csv")  # update path

id_col = "STRUCTURE_NUMBER_008"
year_col = "SURVEY_YEAR"
age_col = "bridge_age"
age_flag_col = "bridge_age_negative_flag"
event_col = "event"

# ---- 1b. Climate: per-year values -> per-bridge "site normals" ----
# Option A keeps the FIRST-event row for failures but the LAST row for censored
# bridges, so per-year climate carried from the kept row fingerprints the
# observation era and leaks censoring status (measured for all three national
# model families in ../Bridges_all_of_US/leakage_ablation_national.ipynb; see the
# root ../README.md "Leakage audit"). Averaging each bridge's climate over its
# whole panel keeps only the spatial exposure signal.
CLIMATE_COLS = [c for c in [
    "freeze_thaw_cycles", "freezing_index", "annual_precip_mm", "snow_days",
    "max_swe", "mean_tmax_DJF", "mean_tmax_JJA", "mean_tmin_DJF", "mean_tmin_JJA",
] if c in df.columns]
df[CLIMATE_COLS] = df.groupby(id_col)[CLIMATE_COLS].transform("mean")
print(f"Converted {len(CLIMATE_COLS)} climate columns to per-bridge normals.")

# ---- 2. Check bridge_age data quality before using it as the time axis ----
n_bad_age = (df[age_flag_col] == 1).sum() if age_flag_col in df.columns else (df[age_col] < 0).sum()
print(f"Rows with negative/bad bridge_age: {n_bad_age} of {len(df)}")
bad_age_structures = df.loc[df[age_col] < 0, id_col].unique()
if len(bad_age_structures) > 0:
    print(f"{len(bad_age_structures)} structures have at least one negative bridge_age row. "
          f"These likely have a bad YEAR_BUILT_027 value -- inspect/fix before modeling.")
    # Example structures to check by hand:
    print(bad_age_structures[:10])

# ---- 3. Sort within each bridge by survey year ----
df = df.sort_values([id_col, year_col]).reset_index(drop=True)

# ---- 4. For each bridge, find the row to keep: ----

df["_cum_event"] = df.groupby(id_col)[event_col].cummax()
df["_is_first_event_row"] = (df[event_col] == 1) & (df["_cum_event"].groupby(df[id_col]).cumsum() == 1)

def pick_row(g):
    event_rows = g[g[event_col] == 1]
    if len(event_rows) > 0:
        return event_rows.iloc[0]   # first time it hit poor
    else:
        return g.iloc[-1]           # last observed survey (censored)

bridge_level = df.groupby(id_col, group_keys=False).apply(pick_row).reset_index(drop=True)
bridge_level = bridge_level.drop(columns=["_cum_event", "_is_first_event_row"])

# ---- 4b. Era-free reconstruction clock ----
# years_since_reconstruction measured at the kept row's SURVEY_YEAR carries the
# same era fingerprint as per-year climate; this prototype anchors it to a fixed
# reference year instead. HISTORICAL NOTE: the national pipeline later found the
# fixed anchor insufficient (the kept-row YEAR_RECONSTRUCTED_106 is itself
# truncated at the kept year) and now measures reconstruction at each bridge's
# PANEL-ENTRY row — see ../Bridges_all_of_US/build_national_rsf_dataset.ipynb,
# deviation 5. This MA prototype is superseded and kept as-is.
REF_YEAR = 2025
yr_recon = pd.to_numeric(bridge_level["YEAR_RECONSTRUCTED_106"], errors="coerce").replace(0, np.nan)
bridge_level["years_since_reconstruction"] = REF_YEAR - yr_recon

# ---- 5. Build final time/event columns ----
bridge_level["time"] = bridge_level[age_col].astype(float)
bridge_level["event"] = bridge_level[event_col].astype(bool)  # sksurv wants boolean event

# ---- 6. Drop rows with unusable time values ----
n_before = len(bridge_level)
bridge_level = bridge_level[bridge_level["time"] > 0]
n_after = len(bridge_level)
print(f"Dropped {n_before - n_after} bridges with non-positive time (bad bridge_age).")

# ---- 7. Quick sanity checks ----
print(f"\nFinal bridge-level dataset: {len(bridge_level)} bridges")
print(f"Events (transitioned to poor): {bridge_level['event'].sum()}")
print(f"Censored (never observed transitioning): {(~bridge_level['event']).sum()}")
print(f"Time range: {bridge_level['time'].min():.1f} to {bridge_level['time'].max():.1f} years")

# ---- 8. Build sksurv structured array target ----
from sksurv.util import Surv
y = Surv.from_dataframe("event", "time", bridge_level)

print("\nSample of final time/event columns:")
print(bridge_level[[id_col, year_col, age_col, "time", "event"]].head(10))

# ---- 9. Save outputs ----
bridge_level.to_csv("RSF_data_a.csv", index=False)
print("\nSaved to RSF_data_a.csv")

# X (features) would then be bridge_level.drop(columns=[id_col, year_col, "time", event_col, "event", ...other leakage cols])
# y is the structured array above, ready for RandomSurvivalForest.fit(X, y)


In [ ]:
#Make df suitable for RSF

import pandas as pd
import numpy as np

df = pd.read_csv("RSF_data_a.csv")

#### Encode 31 ####

# Step 1: clean to consistent string codes
df['design_load_clean'] = (
    df['DESIGN_LOAD_031']
    .astype(str)
    .str.strip()
    .str.replace('.0', '', regex=False)  # '4.0' -> '4'
)

df['design_load_clean'] = df['design_load_clean'].replace({'A': np.nan, 'nan': np.nan})

ordinal_map = {'7': np.nan,'8':np.nan, '1':1, '2':2, '3':3, '4':4, '5':5, '6':6, '9':7, }  # 9->7 to keep contiguous rank
df['design_load_ordinal'] = df['design_load_clean'].map(ordinal_map)

#### Encode TEMP ####
df['TEMP_STRUCTURE_103'] = (df['TEMP_STRUCTURE_103'] == 'T').astype(int)

#### Encode 107 ####

# Recode N to Unknown, collapse rare levels to Other
rare_levels = ['4', '5', '7']

df['DECK_STRUCTURE_TYPE_107'] = df['DECK_STRUCTURE_TYPE_107'].replace('N', 'Unknown')
df['DECK_STRUCTURE_TYPE_107'] = df['DECK_STRUCTURE_TYPE_107'].replace(rare_levels, 'Other')

# One-hot encode
df = pd.get_dummies(df, columns=['DECK_STRUCTURE_TYPE_107'], prefix='DECK_TYPE')

rare_levels = ['4', '8']

df['SURFACE_TYPE_108A'] = df['SURFACE_TYPE_108A'].replace('N', 'Unknown')
df['SURFACE_TYPE_108A'] = df['SURFACE_TYPE_108A'].replace(rare_levels, 'Other')

df = pd.get_dummies(df, columns=['SURFACE_TYPE_108A'], prefix='SURFACE_TYPE')

df['MEMBRANE_TYPE_108B'] = df['MEMBRANE_TYPE_108B'].replace('N', 'Unknown')

df = pd.get_dummies(df, columns=['MEMBRANE_TYPE_108B'], prefix='MEMBRANE_TYPE')
### 108c ###
rare_levels = ['6', '4', '2','3']

df['DECK_PROTECTION_108C'] = df['DECK_PROTECTION_108C'].replace('N', 'Unknown')
df['DECK_PROTECTION_108C'] = df['DECK_PROTECTION_108C'].replace(rare_levels, 'Other')

df = pd.get_dummies(df, columns=['DECK_PROTECTION_108C'], prefix='DECK_PROTECTION')

scour_map = {
    'N': np.nan,   # not over waterway — not missing, but genuinely inapplicable
    'U': np.nan,   # unknown foundation, unscoured
    'T': np.nan,   # tidal, low risk but unevaluated
    '99': np.nan   # miscoded
}

df['SCOUR_CRITICAL_113'] = df['SCOUR_CRITICAL_113'].replace(scour_map).astype(float)

rare_districts = [0.0, 9.0, 27.0]

df['HIGHWAY_DISTRICT_002'] = df['HIGHWAY_DISTRICT_002'].replace(rare_districts, 99.0)
df['HIGHWAY_DISTRICT_002'] = df['HIGHWAY_DISTRICT_002'].astype(str)

df = pd.get_dummies(df, columns=['HIGHWAY_DISTRICT_002'], prefix='HIGHWAY_DISTRICT')

threshold = 50
counts = df['COUNTY_CODE_003'].value_counts()
rare_counties = counts[counts < threshold].index.tolist()

df['COUNTY_CODE_003'] = df['COUNTY_CODE_003'].replace(rare_counties, 99.0)
df['COUNTY_CODE_003'] = df['COUNTY_CODE_003'].astype(str)

df = pd.get_dummies(df, columns=['COUNTY_CODE_003'], prefix='COUNTY')

threshold = 50
counts = df['MAINTENANCE_021'].value_counts()
rare_maintenance = counts[counts < threshold].index.tolist()

df['MAINTENANCE_021'] = df['MAINTENANCE_021'].replace(rare_maintenance, 99.0)
df['MAINTENANCE_021'] = df['MAINTENANCE_021'].astype(str)

df = pd.get_dummies(df, columns=['MAINTENANCE_021'], prefix='MAINTENANCE')

#### 26 #### Broken into binary of rural or not then dummy columns for the types of bridges.
rural_codes = [1.0, 2.0, 6.0, 7.0, 8.0, 9.0]
df['IS_RURAL_026'] = df['FUNCTIONAL_CLASS_026'].isin(rural_codes).astype(int)

road_class_map = {
    1.0: 'Interstate',
    11.0: 'Interstate',
    2.0: 'Principal_Arterial',
    12.0: 'Principal_Arterial',
    14.0: 'Principal_Arterial',
    6.0: 'Minor_Arterial',
    16.0: 'Minor_Arterial',
    7.0: 'Collector',
    17.0: 'Collector',
    8.0: 'Minor_Collector',
    9.0: 'Local',
    19.0: 'Local'
}

df['ROAD_CLASS_026'] = df['FUNCTIONAL_CLASS_026'].map(road_class_map)
df = pd.get_dummies(df, columns=['ROAD_CLASS_026'], prefix='ROAD_CLASS')
df = df.drop(columns=['FUNCTIONAL_CLASS_026'])

#### 37 #### One hot encode the historical designations
df = pd.get_dummies(df, columns = ['HISTORY_037'], prefix='HISTORY')

####42 a #### One hot encode
rare_counts = [6.0, 2.0, 0.0, 4.0, 7.0, 3.0]

df['SERVICE_ON_042A'] = df['SERVICE_ON_042A'].replace(rare_counts, "Other")
df['SERVICE_ON_042A'] = df['SERVICE_ON_042A'].astype(str)

df = pd.get_dummies(df, columns=['SERVICE_ON_042A'], prefix='SERVICE_ON_042A')

####42 b #### One hot encode
rare_counts = [8.0, 9.0]

df['SERVICE_UND_042B'] = df['SERVICE_UND_042B'].replace(rare_counts, "Other")
df['SERVICE_UND_042B'] = df['SERVICE_UND_042B'].astype(str)

df = pd.get_dummies(df, columns=['SERVICE_UND_042B'], prefix='SERVICE_UND_042B')

####43 a####

df = pd.get_dummies(df, columns = ['STRUCTURE_KIND_043A'], prefix='STRUCTURE_KIND_043A')

####43 b####
counts = df['STRUCTURE_TYPE_043B'].value_counts()

rare = counts[counts < 50].index

df['STRUCTURE_TYPE_043B'] = (
    df['STRUCTURE_TYPE_043B']
    .replace(rare, 'Other')
)
df = pd.get_dummies(df, columns = ['STRUCTURE_TYPE_043B'], prefix='STRUCTURE_TYPE_043B')

#### 100 ####
df = pd.get_dummies(df, columns = ['STRAHNET_HIGHWAY_100'], prefix='STRAHNET_HIGHWAY_100')

#### 102 #### TRAFFIC_DIRECTION_102
df = pd.get_dummies(df, columns = ['TRAFFIC_DIRECTION_102'], prefix='TRAFFIC_DIRECTION_102')

#### 111 ####
rare_counts = [5.0, 3.0]

df['PIER_PROTECTION_111'] = df['PIER_PROTECTION_111'].replace(rare_counts, "Other")
df['PIER_PROTECTION_111'] = df['PIER_PROTECTION_111'].astype(str)

df = pd.get_dummies(df, columns=['PIER_PROTECTION_111'], prefix='PIER_PROTECTION_111')

#### 33 #### MEDIAN_CODE_033
df = pd.get_dummies(df, columns=['MEDIAN_CODE_033'], prefix='MEDIAN_CODE_033')

#### 20 #### TOLL_020
df = pd.get_dummies(df, columns=['TOLL_020'], prefix='TOLL_020')

#### 47 #### HORR_CLR_MT_047
df['HORR_CLR_MT_047'] = df['HORR_CLR_MT_047'].replace([99.0, 99.9], np.nan)
  
#### 64 and 66 ####
df['OPERATING_RATING_064'] = df['OPERATING_RATING_064'].replace([99.9], np.nan)

df['INVENTORY_RATING_066'] = df['INVENTORY_RATING_066'].replace([99.9], np.nan)

#### 106 #### YEAR_RECONSTRUCTED_106 is dropped below: years_since_reconstruction
# is anchored to REF_YEAR = 2025 in Build_RSF cell 1 (era-free), which makes the
# raw reconstruction year exactly collinear with it.

### 19 ####
df['DETOUR_KILOS_019'] = df['DETOUR_KILOS_019'].replace([999.0,690.0, 199.0, 311.0], np.nan)

### 28a #### TRAFFIC_LANES_ON_028A
df['TRAFFIC_LANES_ON_028A'] = df['TRAFFIC_LANES_ON_028A'].replace(93.0, np.nan)

#### 49 ##### STRUCTURE_LEN_MT_049
df['STRUCTURE_LEN_MT_049'] = df['STRUCTURE_LEN_MT_049'].replace(20006.5, np.nan)

#### ROADWAY_WIDTH_MT_051 #### 
df['ROADWAY_WIDTH_MT_051'] = df['ROADWAY_WIDTH_MT_051'].replace([999.9], np.nan)

#### DECK_WIDTH_MT_052 #### 
df['DECK_WIDTH_MT_052'] = df['DECK_WIDTH_MT_052'].replace([999.9], np.nan)

###VERT_CLR_UND_054B###
df['VERT_CLR_UND_054B'] = df['VERT_CLR_UND_054B'].replace([99.99, 99.0], np.nan)

###LAT_UND_MT_055B###
df['LAT_UND_MT_055B'] = df['LAT_UND_MT_055B'].replace([99.9], np.nan)

###LEFT_LAT_UND_MT_056###
df['LEFT_LAT_UND_MT_056'] = df['LEFT_LAT_UND_MT_056'].replace([99.9, 99.8], np.nan)




bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

drop_cols = ['STRUCTURE_NUMBER_008', #ID
             'RECORD_TYPE_005A', 
             'CRITICAL_FACILITY_006B',
             'DESIGN_LOAD_031', 
             'design_load_clean', 
             'RAILINGS_036A', 
             'TRANSITIONS_036B',
             'APPR_RAIL_036C',
             'APPR_RAIL_END_036D',
             'NAVIGATION_038',
             'OPEN_CLOSED_POSTED_041', #leakage
             'VERT_CLR_UND_REF_054A',
             'LAT_UND_REF_055A',
             'BRIDGE_LEN_IND_112',
             'PLACE_CODE_004', #cities and towns
             'YEAR_ADT_030', #irrelevant
             'OWNER_022', #almost same as MAINTNENCE_020
             'APPR_KIND_044A', #>95% "other"
             'APPR_TYPE_044B', #>95% "other"
             'DATE_OF_INSPECT_090', #irrelevant
             'DECK_COND_058', #leakage
             'SUPERSTRUCTURE_COND_059', #leakage
             'SUBSTRUCTURE_COND_060', #leakage
             'CHANNEL_COND_061', #leakage
             'CULVERT_COND_062', #leakage
             'VERT_CLR_OVER_MT_053', #95% nan
             'FEDERAL_LANDS_105', #near zero variance
             'DECK_AREA',# 73% nan
             'MIN_VERT_CLR_010', # 95% nan
             'LRS_INV_ROUTE_013A',#~90% nan
             'SUBROUTE_NO_013B',#~90% nan
             'SURVEY_YEAR',# Metadata
             'YEAR_RECONSTRUCTED_106', # collinear with 2025-anchored years_since_reconstruction
             'had_maintenance', #only available for MA, not available for national
             'time_since_last_maintenance',
             'maintenance_in_progress',
             ]

df = df.drop(columns = drop_cols)

df.to_csv('RSF_data_clean.csv')